# CNC Tool Wear & RUL: run the whole project in Colab
Run the cells **in order, one at a time**. When a cell opens a file picker, choose the file named in the cell.

Colab deletes your files when the session resets, so download `results.zip` (cell 7) when training finishes.

## 1. Upload the project zip (choose `cnc-wear-rul.zip`)

In [ ]:
from google.colab import files
up = files.upload()          # choose cnc-wear-rul.zip
print(list(up))

## 2. Unzip, install packages (about 1 minute)

In [ ]:
!unzip -q -o cnc-wear-rul.zip
%cd /content/cnc-wear-rul
!pip install -q shap xgboost pyyaml streamlit plotly
!ls

## 3. Upload the dataset (choose `mill.mat`)
Skip this cell if `data/mill.mat` is already inside your zip (the check below tells you).

In [ ]:
import os, shutil
os.makedirs('data', exist_ok=True)
if not any(f.endswith(('.mat', '.csv')) for f in os.listdir('data')):
    from google.colab import files
    up = files.upload()      # choose mill.mat
    for name in up:
        shutil.move(name, os.path.join('data', name))
print(os.listdir('data'))

## 4. Train and evaluate (all 8 stages, a few minutes)

In [ ]:
!python train.py

## 5. Look at the tables for your report

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('outputs/results_tables.md').read()))

## 6. Look at the figures

In [ ]:
import glob
from IPython.display import Image, display
for f in sorted(glob.glob('outputs/*.png')):
    print(f)
    display(Image(f, width=520))

## 7. Download everything (do this before the session ends)

In [ ]:
!zip -rq results.zip models outputs
from google.colab import files
files.download('results.zip')

## 8. (Optional) Start the dashboard inside Colab
Run cell 8a first and check it prints `HTTP/1.1 200 OK`. Only then run cell 8b and open the `trycloudflare.com` link it prints. The link works only while cell 8b keeps running. For the exam, run the app on your laptop instead.

In [ ]:
import subprocess, time
log = open('/content/log.txt', 'w')
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.address', '0.0.0.0',
                  '--server.headless', 'true'], stdout=log, stderr=subprocess.STDOUT)
time.sleep(10)
!curl -sI http://127.0.0.1:8501 | head -1
!tail -15 /content/log.txt

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://127.0.0.1:8501